In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from utils import DATA_DIR, RANDOM_STATE
from utils.data import CATEGORICAL_COLS
from IPython.display import display

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")

DATA_PATH = DATA_DIR / "hour.csv"
OUT_DIR = DATA_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["dteday"])
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nNumber of duplicates:", df.duplicated().sum())

In [ ]:
expected_hours = pd.date_range(
    start=df["dteday"].min(),
    end=df["dteday"].max() + pd.Timedelta(hours=23),
    freq="h",
)
print("Expected hours:", len(expected_hours))
print("Present rows:", len(df))
print("Missing hours:", len(expected_hours) - len(df))

In [ ]:
leakage_cols = ["casual", "registered"]
id_cols = ["instant"]

df = df.drop(columns=leakage_cols + id_cols)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["cnt"], bins=50)
axes[0].set_title("cnt (raw)")
axes[0].set_xlabel("Rentals per hour")
axes[1].hist(np.log1p(df["cnt"]), bins=50)
axes[1].set_title("log1p(cnt)")
axes[1].set_xlabel("log(1 + rentals)")
plt.tight_layout()
display(fig)

print("Skewness cnt        :", df["cnt"].skew().round(3))
print("Skewness log1p(cnt) :", np.log1p(df["cnt"]).skew().round(3))

In [ ]:
df["cnt_log1p"] = np.log1p(df["cnt"])

In [ ]:
# Remove redundant features:
# workingday is fully derivable from weekday + holiday
# season is fully derivable from mnth (months 3 to 5 = spring and so on)
df = df.drop(columns=["workingday", "season"])

# yr and holiday are binary (0/1) and treated as float64.
# For tree models this is equivalent to category, no information loss.

# Real categorical features (nominal / ordinal groups) from utils.data
numeric_cols = ["temp", "atemp", "hum", "windspeed"]  # atemp dropped later

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("category")

print("Removed features: workingday, season")
print(f"Remaining categorical features: {CATEGORICAL_COLS}")
df.dtypes

In [ ]:
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlations of numeric features")
plt.tight_layout()
display(fig)

In [ ]:
df = df.drop(columns=["atemp"])
numeric_cols = [c for c in numeric_cols if c != "atemp"]
df.head()

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Day level split: all hours of a day go either into train OR test, never both.
# This prevents temporal leakage from neighbouring hours of the same day landing
# in different splits.
unique_days = df["dteday"].unique()  # about 730 unique days

train_days, test_days = train_test_split(
    unique_days,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)

train_df = df[df["dteday"].isin(train_days)].reset_index(drop=True)
test_df  = df[df["dteday"].isin(test_days)].reset_index(drop=True)

print(f"Unique days total : {len(unique_days)}")
print(f"Unique days train : {len(train_days)}")
print(f"Unique days test  : {len(test_days)}")
print(f"Train: {len(train_df)} rows ({len(train_df)/len(df):.1%})")
print(f"Test : {len(test_df)} rows ({len(test_df)/len(df):.1%})")

In [ ]:
feature_cols = [c for c in df.columns if c not in ["dteday", "cnt", "cnt_log1p"]]

X_train = train_df[feature_cols]
y_train = train_df["cnt"]
y_train_log = train_df["cnt_log1p"]

X_test = test_df[feature_cols]
y_test = test_df["cnt"]
y_test_log = test_df["cnt_log1p"]

print("Features:", feature_cols)
print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

In [ ]:
train_df.to_csv(OUT_DIR / "train.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print("Saved to:", OUT_DIR.resolve())